In [1]:
import nltk

nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\SUMIT\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\SUMIT\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\SUMIT\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\universal_tagset.zip.


True

In [2]:
import spacy
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk import pos_tag
nlp = spacy.load("en_core_web_sm")
sentences = [
    "I book a ticket.",
    "She reads a book.",
    "The smart boy plays cricket.",
    "They are playing football.",
    "The weather is beautiful today."
]
for sentence in sentences:
    tokens = word_tokenize(sentence)
    tagged_words = pos_tag(tokens)

    print(f"\nSentence: {sentence}")
    print(tagged_words)


Sentence: I book a ticket.
[('I', 'PRP'), ('book', 'NN'), ('a', 'DT'), ('ticket', 'NN'), ('.', '.')]

Sentence: She reads a book.
[('She', 'PRP'), ('reads', 'VBZ'), ('a', 'DT'), ('book', 'NN'), ('.', '.')]

Sentence: The smart boy plays cricket.
[('The', 'DT'), ('smart', 'JJ'), ('boy', 'NN'), ('plays', 'VBZ'), ('cricket', 'NN'), ('.', '.')]

Sentence: They are playing football.
[('They', 'PRP'), ('are', 'VBP'), ('playing', 'VBG'), ('football', 'NN'), ('.', '.')]

Sentence: The weather is beautiful today.
[('The', 'DT'), ('weather', 'NN'), ('is', 'VBZ'), ('beautiful', 'JJ'), ('today', 'NN'), ('.', '.')]


In [3]:
for sentence in sentences:
    doc = nlp(sentence)

    print(f"\nSentence: {sentence}")

    for token in doc:
        print(token.text, token.pos_, token.tag_)


Sentence: I book a ticket.
I PRON PRP
book VERB VBP
a DET DT
ticket NOUN NN
. PUNCT .

Sentence: She reads a book.
She PRON PRP
reads VERB VBZ
a DET DT
book NOUN NN
. PUNCT .

Sentence: The smart boy plays cricket.
The DET DT
smart ADJ JJ
boy NOUN NN
plays VERB VBZ
cricket NOUN NN
. PUNCT .

Sentence: They are playing football.
They PRON PRP
are AUX VBP
playing VERB VBG
football NOUN NN
. PUNCT .

Sentence: The weather is beautiful today.
The DET DT
weather NOUN NN
is AUX VBZ
beautiful ADJ JJ
today NOUN NN
. PUNCT .


In [4]:
sentence = "The smart boy plays cricket."

tokens = word_tokenize(sentence)

penn_tags = pos_tag(tokens)
universal_tags = pos_tag(tokens, tagset="universal")

comparison = pd.DataFrame({
    "Word": tokens,
    "Penn Treebank": [tag for word, tag in penn_tags],
    "Universal POS": [tag for word, tag in universal_tags]
})

comparison

,Word,Penn Treebank,Universal POS
0,The,DT,DET
1,smart,JJ,ADJ
2,boy,NN,NOUN
3,plays,VBZ,VERB
4,cricket,NN,NOUN
5,.,.,.


In [5]:
comparison = []

for sentence in sentences:
    nltk_tags = pos_tag(word_tokenize(sentence))
    doc = nlp(sentence)

    for (word, nltk_tag), token in zip(nltk_tags, doc):
        comparison.append({
            "Word": word,
            "NLTK": nltk_tag,
            "spaCy": token.tag_,
            "spaCy_POS": token.pos_
        })

comparison_df = pd.DataFrame(comparison)
comparison_df

,Word,NLTK,spaCy,spaCy_POS
0,I,PRP,PRP,PRON
1,book,NN,VBP,VERB
2,a,DT,DT,DET
3,ticket,NN,NN,NOUN
4,.,.,.,PUNCT
5,She,PRP,PRP,PRON
6,reads,VBZ,VBZ,VERB
7,a,DT,DT,DET
8,book,NN,NN,NOUN
9,.,.,.,PUNCT


In [6]:
test_data = [
    ("The", "DET"),
    ("boy", "NOUN"),
    ("plays", "VERB"),
    ("cricket", "NOUN"),
    (".", "."),

    ("She", "PRON"),
    ("is", "VERB"),
    ("reading", "VERB"),
    ("a", "DET"),
    ("book", "NOUN"),
    (".", "."),

    ("They", "PRON"),
    ("are", "VERB"),
    ("happy", "ADJ"),
    (".", "."),

    ("I", "PRON"),
    ("love", "VERB"),
    ("Python", "NOUN"),
    (".", ".")
]
words = [word for word, tag in test_data]
y_true = [tag for word, tag in test_data]

In [7]:
nltk_predictions = pos_tag(words, tagset="universal")

y_pred_nltk = [tag for word, tag in nltk_predictions]

In [8]:
text = "The boy plays cricket. She is reading a book. They are happy. I love Python."

doc = nlp(text)

spacy_predictions = [(token.text, token.pos_) for token in doc]

y_pred_spacy = [tag for word, tag in spacy_predictions]

In [9]:
assert words == [word for word, tag in nltk_predictions]
assert words == [word for word, tag in spacy_predictions]

In [10]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("NLTK Accuracy:", accuracy_score(y_true, y_pred_nltk))
print("spaCy Accuracy:", accuracy_score(y_true, y_pred_spacy))

NLTK Accuracy: 1.0
spaCy Accuracy: 0.631578947368421


In [11]:
print("NLTK Evaluation")
print(classification_report(y_true, y_pred_nltk, zero_division=0))

print("spaCy Evaluation")
print(classification_report(y_true, y_pred_spacy, zero_division=0))

NLTK Evaluation
              precision    recall  f1-score   support

           .       1.00      1.00      1.00         4
         ADJ       1.00      1.00      1.00         1
         DET       1.00      1.00      1.00         2
        NOUN       1.00      1.00      1.00         4
        PRON       1.00      1.00      1.00         3
        VERB       1.00      1.00      1.00         5

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19

spaCy Evaluation
              precision    recall  f1-score   support

           .       0.00      0.00      0.00         4
         ADJ       1.00      1.00      1.00         1
         AUX       0.00      0.00      0.00         0
         DET       1.00      1.00      1.00         2
        NOUN       1.00      0.75      0.86         4
        PRON       1.00      1.00      1.00         3
       PROPN       0.00      0.00      0.00  